# Phase 2 — Parallel MARL GPU Training on Google Colab (Subprocess-Safe)

This notebook automates the training of PPO Multi-Agent Reinforcement Learning (MARL) models using Google Colab's GPU in **parallel CLI processes** to maximize resource utilization and speed up the training process without any multiprocessing crashes.

### Step-by-Step Instructions:
1. **Upload your code**: Zip your workspace folder (exclude the `models` folder to keep it small) and drag-and-drop the `.zip` file into Colab's file pane on the left.
2. **Select GPU Runtime**: Click **Runtime** -> **Change runtime type** -> select **T4 GPU** -> **Save**.
3. **Run all cells** sequentially.

### 1. Extract Workspace Code

In [ ]:
# Locate and unzip your uploaded repository zip file
import os
from pathlib import Path

zip_name = "wildfire-rl.zip"  # Change this if your uploaded zip file has a different name

if Path(zip_name).exists():
    !unzip -q {zip_name} -d wildfire-rl
    %cd wildfire-rl
    print(f"Successfully unzipped and entered directory: {os.getcwd()}")
else:
    print(f"ERROR: Could not find '{zip_name}' in the root directory. Please upload it first via the file pane.")

### 2. Install Project Dependencies

In [ ]:
# Install core packages, stable-baselines3, and register the package in editable mode
# Exclude PyTorch and NumPy to keep Colab's pre-configured GPU runtime active
!grep -v "torch" requirements.txt | grep -v "numpy" > req_clean.txt
!pip install -q -r req_clean.txt
!pip install -q -e .

### 3. Verify CUDA (GPU) Activation

*Note: If you just ran the install cell and got a NumPy error in the next steps, click **Runtime** -> **Restart session** in the top menu, then skip the install cell and resume from here.*

In [ ]:
import torch
print(f"CUDA (GPU) Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not active. Go to Runtime -> Change runtime type and select T4 GPU.")

### 4. Run Parallel Subprocess PPO MARL Training on GPU (100k Steps Matched Budget)

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

FORCE_RETRAIN = True  # Set to True to rerun and overwrite all models
total_timesteps = 100000  # Strict scientific budget-match
max_parallel = 4  # Running 4 models concurrently

# Create all training jobs
jobs = []
for region_name in ["saudi", "california"]:
    for n_agents in [1, 3, 5, 10]:
        for seed in [0, 1, 2]:
            jobs.append((region_name, n_agents, seed))

processes = []
print(f"Launching {len(jobs)} training runs in parallel using {max_parallel} subprocesses...")

for region, agents, seed in jobs:
    model_file = Path("models") / f"ppo_marl_{region}_{agents}agents_seed_{seed}.zip"
    if model_file.exists() and not FORCE_RETRAIN:
        print(f"Skip: {model_file.name} already exists.")
        continue
        
    # Wait if we already have max_parallel processes running
    while len(processes) >= max_parallel:
        for p in list(processes):
            if p.poll() is not None:  # Process finished
                processes.remove(p)
        time.sleep(1)

    # Start an independent OS-level subprocess
    cmd = [
        "python", "scripts/train_single_marl.py",
        "--region", region,
        "--agents", str(agents),
        "--seed", str(seed),
        "--timesteps", str(total_timesteps)
    ]
    print(f"Launching: {' '.join(cmd)}")
    p = subprocess.Popen(cmd)
    processes.append(p)

# Wait for all remaining processes to complete
for p in processes:
    p.wait()

print("\nAll parallel training runs finished successfully!")

### 5. Compress and Download Trained Models

In [ ]:
# Package the models into a single zip file for downloading
!zip -j trained_marl_models.zip models/ppo_marl_*

print("\n--- READY FOR DOWNLOAD ---")
print("1. Click the file explorer icon in Colab (left panel).")
print("2. Click the three dots next to 'trained_marl_models.zip' and select 'Download'.")
print("3. Extract these models locally into your 'models/' folder.")